# AEZ-26 Phase7A Extended Historical Robustness — ERA5-Land / CHIRPS, 1981–2025

This notebook is an **additive sensitivity branch**. It does **not** replace or retune the locked GLDAS 2000–2025 primary catalogue.

It separates two questions:

1. **Source robustness:** ERA5-Land vs GLDAS over the identical 2000–2025 climatology.
2. **Record-length robustness:** ERA5-Land using the longer 1981–2025 same-pentad climatology.

Locked Phase-2C v3 event logic is reused:
- >40th-percentile anchor;
- first subsequent <=20th-percentile crossing;
- mean decline >=5 percentile points/pentad;
- minimum event duration 3 pentads;
- recovery at >=20th percentile;
- old anchor is never reused across a recovered failed dry excursion.

## Critical input guard added after independent audit
The locked Phase-2C v3 primary chronology contains **39 events**. Because a stale/superseded same-named 40-event CSV may coexist in Drive, this notebook **does not trust an arbitrary event-catalogue file returned by `rglob()`**. Instead it:

- locks the authoritative GLDAS pentad table by SHA256;
- reconstructs the Phase-2C v3 primary chronology directly from that pentad table with the frozen first-crossing detector;
- asserts exactly 39 reconstructed primary events;
- audits any same-named event-catalogue candidates and ignores candidates that do not reproduce the locked 39-event chronology.

This prevents a stale catalogue copy from contaminating the Phase7A source-agreement statistics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, hashlib, platform, shutil
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr, kendalltau, theilslopes
import matplotlib.pyplot as plt

BASE = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE7 = BASE / 'Phase7'
OUT = PHASE7 / 'Phase7A_FINAL'
RAW = OUT / 'GEE_Chunks'
FIG = OUT / 'Figures'
for d in (PHASE7, OUT, RAW, FIG):
    d.mkdir(parents=True, exist_ok=True)

# Locked hash of the Phase-2C v3 pentad table that independently
# reconstructs the 39-event first-crossing catalogue.
LOCKED_GLDAS_PENTAD_SHA256 = (
    '588e505b9d33769f9146c289e75edda1948eb80e00d57b100b9f1776fc72e4b4'
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def locate_locked_gldas_pentad():
    name = 'AEZ26_Pentad_Hydroclimate_Indicators_2000_2025.csv'
    hits = list(BASE.rglob(name))
    if not hits:
        raise FileNotFoundError(name)

    exact = [p for p in hits if sha256_file(p) == LOCKED_GLDAS_PENTAD_SHA256]
    if len(exact) == 0:
        raise RuntimeError(
            'No GLDAS pentad-table copy matches the locked Phase-2C v3 SHA256. '
            'Do not continue with an unverified substitute.'
        )
    if len(exact) > 1:
        print('Multiple byte-identical locked GLDAS pentad copies found; using:', exact[0])
    return exact[0], hits

def stage_phase7a_chunks():
    expected = [
        f'AEZ26_Phase7A_ERA5Land_CHIRPS_Pentad_{start}_{start+4}.csv'
        for start in range(1981, 2022, 5)
    ]
    staged = []
    for name in expected:
        preferred = RAW / name
        if preferred.exists():
            staged.append(preferred)
            continue

        hits = [p for p in BASE.rglob(name) if p != preferred]
        if not hits:
            raise FileNotFoundError(
                f'Missing Phase7A GEE chunk: {name}. Run all 9 GEE export tasks first.'
            )
        if len(hits) > 1:
            raise RuntimeError(
                f'Multiple copies found for {name}: {hits}. Keep one authoritative copy.'
            )

        src = hits[0]
        print(f'Moving GEE chunk into Phase7A_FINAL/GEE_Chunks: {src.name}')
        shutil.move(str(src), str(preferred))
        staged.append(preferred)

    return staged

ERA_FILES = stage_phase7a_chunks()
GLDAS_PENTAD_FILE, GLDAS_PENTAD_CANDIDATES = locate_locked_gldas_pentad()

# These files are diagnostics only. They are NOT used as the primary
# chronology unless they independently match the reconstructed v3 onsets.
GLDAS_EVENT_CANDIDATES = list(
    BASE.rglob('AEZ26_Flash_Drought_Event_Catalogue_Primary_2000_2025.csv')
)

print('Phase7A folder:', OUT)
print('ERA robustness chunks:')
for f in ERA_FILES:
    print(' -', f)
print('Locked GLDAS pentad table:', GLDAS_PENTAD_FILE)
print('Same-named GLDAS event-catalogue candidates found:', len(GLDAS_EVENT_CANDIDATES))


In [ ]:
era_parts = []
for f in ERA_FILES:
    part = pd.read_csv(
        f,
        parse_dates=['pentad_start','pentad_end','pentad_mid_date']
    )
    part['source_chunk_file'] = f.name
    era_parts.append(part)

era = (
    pd.concat(era_parts, ignore_index=True)
      .sort_values(['year','pentad_of_year'])
      .reset_index(drop=True)
)

for c in ['year', 'pentad_of_year', 'n_days']:
    era[c] = era[c].astype(int)

g_pentad = pd.read_csv(
    GLDAS_PENTAD_FILE,
    parse_dates=['pentad_start','pentad_end','pentad_mid_date']
).sort_values(['year','pentad_of_year']).reset_index(drop=True)

for c in ['year', 'pentad_of_year', 'n_days']:
    if c in g_pentad.columns:
        g_pentad[c] = g_pentad[c].astype(int)

core = [
    'ERA5_SM_0_100_mean_mm','precipitation_sum_mm','t2m_mean_C',
    'VPD_mean_kPa','ERA5_potential_evaporation_sum_mm'
]

assert len(ERA_FILES) == 9
assert len(era) == 45*73, len(era)
assert era['year'].min() == 1981 and era['year'].max() == 2025
assert era.groupby('year')['pentad_of_year'].nunique().eq(73).all()
assert not era.duplicated(['year','pentad_of_year']).any()

# Locked GLDAS structural gate.
assert len(g_pentad) == 26*73, len(g_pentad)
assert g_pentad['year'].min() == 2000 and g_pentad['year'].max() == 2025
assert g_pentad.groupby('year')['pentad_of_year'].nunique().eq(73).all()
assert not g_pentad.duplicated(['year','pentad_of_year']).any()
assert g_pentad['SM_0_100_pct'].notna().all()

# Robust boolean handling for GEE CSV exports.
complete = era['complete_pentad_flag']
if complete.dtype == object:
    complete = complete.astype(str).str.lower().map(
        {'true':1,'false':0,'1':1,'0':0}
    )
else:
    complete = complete.astype(int)

assert complete.eq(1).all(), era.loc[
    complete.ne(1),
    ['year','pentad_of_year','era5_daily_images','chirps_daily_images']
].head(20)

assert era[core].isna().sum().sum() == 0, era[core].isna().sum()

# Expected chunk coverage: 5 years × 73 pentads = 365 rows each.
chunk_counts = era.groupby('source_chunk_file').size()
assert len(chunk_counts) == 9
assert chunk_counts.eq(365).all(), chunk_counts

# Daily-image counts must exactly equal the actual number of days in each pentad.
assert (era['era5_daily_images'].astype(int) == era['n_days']).all()
assert (era['chirps_daily_images'].astype(int) == era['n_days']).all()

print('ERA5-Land robustness extraction QC: PASS')
print('Rows:', len(era))
print('Chunks:', len(ERA_FILES))
print(chunk_counts.to_string())
print('Locked GLDAS pentad QC: PASS')


## Percentile branches

- `ERA5_SM_pct_LONG45`: same-pentad empirical percentile from 1981–2025.
- `ERA5_SM_pct_OVERLAP26`: same-pentad empirical percentile using only 2000–2025.

The second branch isolates **soil-moisture-source sensitivity** from the effect of extending the climatological reference.


In [ ]:
# 45-year long-record percentile.
era['ERA5_SM_pct_LONG45'] = (
    era.groupby('pentad_of_year')['ERA5_SM_0_100_mean_mm']
       .rank(method='average', pct=True) * 100.0
)

# 26-year overlap percentile, defined only for 2000–2025.
era['ERA5_SM_pct_OVERLAP26'] = np.nan
mask = era['year'].between(2000, 2025)
era.loc[mask, 'ERA5_SM_pct_OVERLAP26'] = (
    era.loc[mask]
       .groupby('pentad_of_year')['ERA5_SM_0_100_mean_mm']
       .rank(method='average', pct=True) * 100.0
)

assert era['ERA5_SM_pct_LONG45'].notna().all()
assert era.loc[mask,'ERA5_SM_pct_OVERLAP26'].notna().all()

# Same-pentad standardized anomalies for continuous-state comparison.
era['ERA5_SM_z_OVERLAP26'] = np.nan
ov = era.loc[mask].copy()
stats = ov.groupby('pentad_of_year')['ERA5_SM_0_100_mean_mm'].agg(['mean','std'])
era.loc[mask,'ERA5_SM_z_OVERLAP26'] = [
    (r.ERA5_SM_0_100_mean_mm - stats.loc[r.pentad_of_year,'mean']) /
    stats.loc[r.pentad_of_year,'std']
    for r in era.loc[mask].itertuples()
]

era.to_csv(OUT/'AEZ26_Robustness_ERA5Land_Pentad_Indicators_1981_2025.csv',
           index=False, date_format='%Y-%m-%d')
print('Percentile construction: PASS')


In [ ]:
def detect_flash_droughts(
    pentad_df, pct_col, catalogue_name,
    start_threshold=40.0, drought_threshold=20.0,
    recovery_threshold=20.0, min_decline_rate=5.0,
    min_event_duration_pentads=3
):
    x = pentad_df.sort_values('pentad_start').reset_index(drop=True).copy()
    pct = x[pct_col].to_numpy(dtype=float)
    events = []
    i = 0
    event_number = 1
    n = len(x)

    while i < n - 1:
        if not np.isfinite(pct[i]) or pct[i] <= start_threshold:
            i += 1
            continue

        j = i + 1
        episode_handled = False

        while j < n:
            if not np.isfinite(pct[j]):
                i = j + 1
                episode_handled = True
                break

            # New >40 value supersedes the older anchor.
            if pct[j] > start_threshold:
                i = j
                episode_handled = True
                break

            # Evaluate the FIRST <=20 crossing only.
            if pct[j] <= drought_threshold:
                decline_rate = (pct[i] - pct[j]) / (j - i)

                recovery_idx = j + 1
                while (
                    recovery_idx < n and
                    np.isfinite(pct[recovery_idx]) and
                    pct[recovery_idx] < recovery_threshold
                ):
                    recovery_idx += 1

                right_censored = (
                    recovery_idx >= n or
                    (recovery_idx < n and not np.isfinite(pct[recovery_idx]))
                )
                event_end_idx = n-1 if right_censored else recovery_idx-1
                onset_idx = i + 1
                duration = event_end_idx - onset_idx + 1

                qualifies = (
                    decline_rate >= min_decline_rate and
                    duration >= min_event_duration_pentads
                )

                if qualifies:
                    ev = x.loc[onset_idx:event_end_idx].copy()
                    ant = x.loc[max(0,onset_idx-4):onset_idx-1].copy()

                    events.append({
                        'catalogue': catalogue_name,
                        'event_id': f'{catalogue_name}_FD_{event_number:03d}',
                        'anchor_pentad': x.loc[i,'pentad_label'],
                        'anchor_percentile': float(pct[i]),
                        'onset_pentad': x.loc[onset_idx,'pentad_label'],
                        'onset_start_date': x.loc[onset_idx,'pentad_start'],
                        'onset_mid_date': x.loc[onset_idx,'pentad_mid_date'],
                        'threshold_crossing_pentad': x.loc[j,'pentad_label'],
                        'threshold_crossing_percentile': float(pct[j]),
                        'event_end_pentad': x.loc[event_end_idx,'pentad_label'],
                        'recovery_pentad': (
                            None if right_censored
                            else x.loc[recovery_idx,'pentad_label']
                        ),
                        'right_censored_flag': int(right_censored),
                        'intensification_duration_pentads': int(j-i),
                        'total_duration_pentads': int(duration),
                        'total_duration_actual_days': int(ev['n_days'].sum()),
                        'mean_intensification_rate_pp_per_pentad': float(decline_rate),
                        'minimum_soil_moisture_percentile': float(ev[pct_col].min()),
                        'severity_40pct_deficit_pp_pentads': float(
                            np.maximum(
                                0.0,
                                40.0-ev[pct_col].to_numpy(float)
                            ).sum()
                        ),
                        'onset_year': int(x.loc[onset_idx,'year']),
                        'onset_month': int(x.loc[onset_idx,'pentad_mid_date'].month),
                        'antecedent_4pentad_precipitation_mm': (
                            float(ant['precipitation_sum_mm'].sum())
                            if len(ant) and 'precipitation_sum_mm' in ant.columns
                            else np.nan
                        ),
                        'event_precipitation_mm': (
                            float(ev['precipitation_sum_mm'].sum())
                            if 'precipitation_sum_mm' in ev.columns
                            else np.nan
                        ),
                        'event_VPD_mean_kPa': (
                            float(np.average(
                                ev['VPD_mean_kPa'], weights=ev['n_days']
                            ))
                            if 'VPD_mean_kPa' in ev.columns
                            else np.nan
                        ),
                        'event_t2m_mean_C': (
                            float(np.average(
                                ev['t2m_mean_C'], weights=ev['n_days']
                            ))
                            if 't2m_mean_C' in ev.columns
                            else np.nan
                        ),
                        'event_ERA5_potential_evaporation_mm': (
                            float(ev['ERA5_potential_evaporation_sum_mm'].sum())
                            if 'ERA5_potential_evaporation_sum_mm' in ev.columns
                            else np.nan
                        )
                    })
                    event_number += 1

                # Accept or reject, consume through recovery; never reuse old anchor.
                i = n if right_censored else recovery_idx
                episode_handled = True
                break

            j += 1

        if not episode_handled:
            i += 1

    return pd.DataFrame(events)

# ------------------------------------------------------------------
# Reconstruct the LOCKED Phase-2C v3 GLDAS chronology from the locked
# pentad table. This is the source used for Phase7A agreement statistics.
# ------------------------------------------------------------------
g_events = detect_flash_droughts(
    g_pentad,
    'SM_0_100_pct',
    'PRIMARY_SM0_100'
)
assert len(g_events) == 39, (
    f'Locked Phase-2C v3 reconstruction should contain 39 events, '
    f'but found {len(g_events)}.'
)

g_events.to_csv(
    OUT/'AEZ26_Phase7A_Audit_Reconstructed_GLDAS_v3_Primary_39.csv',
    index=False,
    date_format='%Y-%m-%d'
)

# Audit any same-named event-catalogue files in Drive, but do not trust
# them automatically. A 40-event stale/superseded copy is explicitly ignored.
candidate_rows = []
locked_onsets = g_events['onset_pentad'].tolist()

for f in GLDAS_EVENT_CANDIDATES:
    try:
        tmp = pd.read_csv(f)
        onsets = tmp['onset_pentad'].astype(str).tolist()
        exact = (len(tmp) == 39 and onsets == locked_onsets)
        candidate_rows.append({
            'path': str(f),
            'rows': len(tmp),
            'sha256': sha256_file(f),
            'matches_reconstructed_v3_39_exactly': bool(exact),
            'status': 'LOCKED_MATCH' if exact else 'STALE_OR_SUPERSEDED_IGNORE'
        })
    except Exception as exc:
        candidate_rows.append({
            'path': str(f),
            'rows': np.nan,
            'sha256': sha256_file(f),
            'matches_reconstructed_v3_39_exactly': False,
            'status': f'UNREADABLE_IGNORE: {exc}'
        })

candidate_audit = pd.DataFrame(candidate_rows)
candidate_audit.to_csv(
    OUT/'AEZ26_Phase7A_GLDAS_Primary_Candidate_Audit.csv',
    index=False
)

# Source-only robustness: same 2000–2025 climatological length as GLDAS.
era26 = era.loc[era['year'].between(2000,2025)].copy()
cat_source = detect_flash_droughts(
    era26, 'ERA5_SM_pct_OVERLAP26', 'ERA5_SOURCE26'
)

# Long-record robustness: full 1981–2025 climatology.
cat_long = detect_flash_droughts(
    era, 'ERA5_SM_pct_LONG45', 'ERA5_LONG45'
)

cat_source.to_csv(
    OUT/'AEZ26_Robustness_ERA5Land_Source26_Event_Catalogue_2000_2025.csv',
    index=False, date_format='%Y-%m-%d'
)
cat_long.to_csv(
    OUT/'AEZ26_Robustness_ERA5Land_Long45_Event_Catalogue_1981_2025.csv',
    index=False, date_format='%Y-%m-%d'
)

print('LOCKED GLDAS v3 reconstructed primary:', len(g_events))
print('ERA5 source-only 2000–2025:', len(cat_source))
print('ERA5 long-record 1981–2025:', len(cat_long))
print(
    'ERA5 long-record onsets in 2000–2025:',
    int(cat_long['onset_year'].between(2000,2025).sum())
)
display(candidate_audit)


In [ ]:
# Independent event-logic audit.
def audit_catalogue(cat, pentad, pct_col):
    p = pentad.set_index('pentad_label')
    violations = {
        'anchor_not_gt40':0,
        'cross_not_le20':0,
        'rate_lt5':0,
        'duration_lt3':0,
        'first_crossing_violation':0,
        'event_overlap':0
    }
    intervals = []
    labels = pentad['pentad_label'].tolist()
    pos = {v:i for i,v in enumerate(labels)}

    for ev in cat.itertuples(index=False):
        ai = pos[ev.anchor_pentad]
        ci = pos[ev.threshold_crossing_pentad]
        oi = pos[ev.onset_pentad]
        ei = pos[ev.event_end_pentad]

        violations['anchor_not_gt40'] += int(p.loc[ev.anchor_pentad,pct_col] <= 40)
        violations['cross_not_le20'] += int(p.loc[ev.threshold_crossing_pentad,pct_col] > 20)
        violations['rate_lt5'] += int(ev.mean_intensification_rate_pp_per_pentad < 5)
        violations['duration_lt3'] += int(ev.total_duration_pentads < 3)

        between = pentad.loc[ai+1:ci-1, pct_col].to_numpy(float)
        violations['first_crossing_violation'] += int(np.any(between <= 20))
        intervals.append((oi,ei))

    for a,b in zip(intervals[:-1],intervals[1:]):
        violations['event_overlap'] += int(b[0] <= a[1])

    return violations

audit_source = audit_catalogue(cat_source, era26.reset_index(drop=True), 'ERA5_SM_pct_OVERLAP26')
audit_long = audit_catalogue(cat_long, era.reset_index(drop=True), 'ERA5_SM_pct_LONG45')

print('Source catalogue audit:', audit_source)
print('Long catalogue audit:', audit_long)
assert all(v == 0 for v in audit_source.values())
assert all(v == 0 for v in audit_long.values())
print('Event logic audit: PASS')


## One-to-one onset agreement with the locked GLDAS catalogue

Matching is done in pentad sequence, without allowing one sensitivity event to match multiple primary events.


In [ ]:
def ordinal_from_label(s):
    y = int(str(s).split('_P')[0])
    p = int(str(s).split('_P')[1])
    return y*73 + p

def one_to_one_match(primary, sensitivity, tolerance):
    A = [(i, ordinal_from_label(v)) for i,v in enumerate(primary['onset_pentad'])]
    B = [(j, ordinal_from_label(v)) for j,v in enumerate(sensitivity['onset_pentad'])]
    candidates = []
    for i,a in A:
        for j,b in B:
            d = abs(a-b)
            if d <= tolerance:
                candidates.append((d,i,j))
    candidates.sort()
    usedA, usedB, pairs = set(), set(), []
    for d,i,j in candidates:
        if i not in usedA and j not in usedB:
            usedA.add(i); usedB.add(j)
            pairs.append((i,j,d))
    return pairs

# Source-only comparison is the cleanest GLDAS-vs-ERA5 test.
agreement_rows = []
for tol in [0,1,2]:
    pairs = one_to_one_match(g_events, cat_source, tol)
    agreement_rows.append({
        'comparison':'GLDAS_primary_vs_ERA5_source26',
        'tolerance_pentads':tol,
        'matched_primary_events':len(pairs),
        'primary_total':len(g_events),
        'sensitivity_total':len(cat_source),
        'primary_match_fraction':len(pairs)/len(g_events)
    })

# Combined source + long-climatology effect, restricted to 2000–2025.
cat_long_overlap = cat_long.loc[cat_long['onset_year'].between(2000,2025)].reset_index(drop=True)
for tol in [0,1,2]:
    pairs = one_to_one_match(g_events, cat_long_overlap, tol)
    agreement_rows.append({
        'comparison':'GLDAS_primary_vs_ERA5_long45_overlap',
        'tolerance_pentads':tol,
        'matched_primary_events':len(pairs),
        'primary_total':len(g_events),
        'sensitivity_total':len(cat_long_overlap),
        'primary_match_fraction':len(pairs)/len(g_events)
    })

agreement = pd.DataFrame(agreement_rows)
display(agreement)
agreement.to_csv(OUT/'AEZ26_Robustness_Event_Onset_Agreement.csv', index=False)


In [ ]:
# Continuous-state overlap comparison.
merged = g_pentad[
    ['pentad_label','SM_0_100cm_mean_mm','SM_0_100_pct']
].merge(
    era26[['pentad_label','ERA5_SM_0_100_mean_mm',
           'ERA5_SM_pct_OVERLAP26','ERA5_SM_z_OVERLAP26']],
    on='pentad_label', how='inner'
)

# GLDAS same-pentad standardized anomaly.
gg = g_pentad.copy()
gstats = gg.groupby('pentad_of_year')['SM_0_100cm_mean_mm'].agg(['mean','std'])
gg['GLDAS_SM_z'] = [
    (r.SM_0_100cm_mean_mm-gstats.loc[r.pentad_of_year,'mean'])/
    gstats.loc[r.pentad_of_year,'std']
    for r in gg.itertuples()
]
merged = merged.merge(gg[['pentad_label','GLDAS_SM_z']], on='pentad_label')

rho_pct, p_pct = spearmanr(
    merged['SM_0_100_pct'], merged['ERA5_SM_pct_OVERLAP26']
)
rho_z, p_z = spearmanr(
    merged['GLDAS_SM_z'], merged['ERA5_SM_z_OVERLAP26']
)

state_agreement = pd.DataFrame([
    {'metric':'Spearman percentile agreement','value':rho_pct,'p_value':p_pct},
    {'metric':'Spearman standardized-anomaly agreement','value':rho_z,'p_value':p_z},
])
display(state_agreement)
state_agreement.to_csv(OUT/'AEZ26_Robustness_Continuous_State_Agreement.csv', index=False)

plt.figure(figsize=(6,6))
plt.scatter(merged['SM_0_100_pct'], merged['ERA5_SM_pct_OVERLAP26'], s=8, alpha=.45)
plt.xlabel('GLDAS 0–100 cm percentile')
plt.ylabel('ERA5-Land 0–100 cm-equivalent percentile')
plt.title('AEZ-26 soil-moisture percentile agreement, 2000–2025')
plt.tight_layout()
plt.savefig(FIG/'AEZ26_Robustness_GLDAS_ERA5_Percentile_Agreement.png', dpi=250)
plt.show()


## Long-record climatology and hydroclimate context

Trend diagnostics are **robustness/context evidence**, not a replacement for the 2000–2025 primary event inventory.
Theil–Sen slopes and Kendall tests are reported transparently; trend language should remain restrained where p-values or source sensitivity do not support a stable signal.


In [ ]:
# Annual event counts including zero-event years.
years = pd.DataFrame({'year':np.arange(1981,2026)})
annual_fd = cat_long.groupby('onset_year').size().rename('flash_drought_count').reset_index()
annual_fd = years.merge(annual_fd, left_on='year', right_on='onset_year', how='left')
annual_fd['flash_drought_count'] = annual_fd['flash_drought_count'].fillna(0).astype(int)
annual_fd = annual_fd.drop(columns='onset_year')

def trend_row(year, values, name, multiplier=10.0, unit='per decade'):
    year = np.asarray(year, float)
    values = np.asarray(values, float)
    slope, intercept, lo, hi = theilslopes(values, year, 0.95)
    tau, p = kendalltau(year, values)
    return {
        'metric':name,
        'theil_sen_change_per_decade':slope*multiplier,
        'theil_sen_95ci_low_per_decade':lo*multiplier,
        'theil_sen_95ci_high_per_decade':hi*multiplier,
        'kendall_tau':tau,
        'kendall_p_value':p,
        'unit':unit
    }

trend_rows = [
    trend_row(annual_fd['year'], annual_fd['flash_drought_count'],
              'ERA5-Land flash-drought annual count', unit='events/decade')
]

# Annual hydroclimate metrics.
annual_clim = pd.DataFrame([
    {
        'year': int(y),
        'temperature_C': np.average(g['t2m_mean_C'], weights=g['n_days']),
        'VPD_kPa': np.average(g['VPD_mean_kPa'], weights=g['n_days']),
        'precipitation_mm': g['precipitation_sum_mm'].sum(),
        'potential_evaporation_mm': g['ERA5_potential_evaporation_sum_mm'].sum()
    }
    for y, g in era.groupby('year', sort=True)
])

trend_rows += [
    trend_row(annual_clim['year'], annual_clim['temperature_C'],
              'Annual mean temperature', unit='degC/decade'),
    trend_row(annual_clim['year'], annual_clim['VPD_kPa'],
              'Annual mean VPD', unit='kPa/decade'),
    trend_row(annual_clim['year'], annual_clim['precipitation_mm'],
              'Annual CHIRPS precipitation', unit='mm/decade'),
    trend_row(annual_clim['year'], annual_clim['potential_evaporation_mm'],
              'Annual ERA5-Land potential evaporation', unit='mm/decade')
]

trends = pd.DataFrame(trend_rows)
display(trends)
trends.to_csv(OUT/'AEZ26_Robustness_Long_Record_Trend_Diagnostics.csv', index=False)

annual_fd.to_csv(OUT/'AEZ26_Robustness_Annual_FD_Counts_1981_2025.csv', index=False)
annual_clim.to_csv(OUT/'AEZ26_Robustness_Annual_Hydroclimate_1981_2025.csv', index=False)


In [ ]:
# Period and season summaries.
def season(m):
    if m in [12,1,2]: return 'Winter_DJF'
    if m in [3,4,5]: return 'PreMonsoon_MAM'
    if m in [6,7,8,9]: return 'Monsoon_JJAS'
    return 'PostMonsoon_ON'

cat_long['season'] = cat_long['onset_month'].map(season)
cat_long['period'] = pd.cut(
    cat_long['onset_year'],
    bins=[1980,1995,2010,2025],
    labels=['1981-1995','1996-2010','2011-2025']
)

season_order = [
    'Winter_DJF',
    'PreMonsoon_MAM',
    'Monsoon_JJAS',
    'PostMonsoon_ON'
]
season_summary = (
    cat_long.groupby('season').size()
    .reindex(season_order, fill_value=0)
    .rename('events')
    .reset_index()
)

period_summary = (
    cat_long.groupby('period', observed=False)
    .size()
    .rename('events')
    .reset_index()
)
period_summary['years'] = 15
period_summary['events_per_year'] = period_summary['events']/15

display(season_summary)
display(period_summary)

season_summary.to_csv(
    OUT/'AEZ26_Robustness_Long45_Seasonality.csv',
    index=False
)
period_summary.to_csv(
    OUT/'AEZ26_Robustness_Long45_Period_Event_Rates.csv',
    index=False
)

plt.figure(figsize=(10,4.5))
plt.bar(annual_fd['year'], annual_fd['flash_drought_count'])
plt.xlabel('Year')
plt.ylabel('Regional flash-drought onsets')
plt.title('ERA5-Land long-record sensitivity catalogue, 1981–2025')
plt.tight_layout()
plt.savefig(
    FIG/'AEZ26_Robustness_Annual_FD_Counts_1981_2025.png',
    dpi=250
)
plt.show()


In [ ]:
# Compact final QC / interpretation gate.
stale_candidates = (
    int((candidate_audit['status'] != 'LOCKED_MATCH').sum())
    if len(candidate_audit) else 0
)

qc = pd.DataFrame([
    ['ERA5/CHIRPS pentad rows', len(era), 45*73, 'PASS'],
    ['Years', era['year'].nunique(), 45, 'PASS'],
    ['Pentads per year minimum',
     era.groupby('year')['pentad_of_year'].nunique().min(), 73, 'PASS'],
    ['Incomplete pentads', int((complete==0).sum()), 0, 'PASS'],
    ['Missing core values', int(era[core].isna().sum().sum()), 0, 'PASS'],
    ['Locked GLDAS v3 reconstructed events', len(g_events), 39, 'PASS'],
    ['Source-only detector logic violations',
     sum(audit_source.values()), 0, 'PASS'],
    ['Long-record detector logic violations',
     sum(audit_long.values()), 0, 'PASS'],
], columns=['check','value','expected','status'])

qc.to_csv(OUT/'AEZ26_Robustness_QC.csv', index=False)
display(qc)

# Reproducibility metadata.
environment = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'ERA5_chunk_sha256': {f.name: sha256_file(f) for f in ERA_FILES},
    'GLDAS_pentad_sha256': sha256_file(GLDAS_PENTAD_FILE),
    'GLDAS_pentad_locked_expected_sha256': LOCKED_GLDAS_PENTAD_SHA256,
    'GLDAS_primary_for_agreement':
        'reconstructed_from_locked_pentad_with_Phase2C_v3_first_crossing_logic',
    'GLDAS_reconstructed_primary_event_count': int(len(g_events)),
    'same_named_event_catalogue_candidates_inspected':
        int(len(GLDAS_EVENT_CANDIDATES)),
    'same_named_stale_or_superseded_candidates_ignored': stale_candidates,
    'GLDAS_event_candidate_audit_file':
        'AEZ26_Phase7A_GLDAS_Primary_Candidate_Audit.csv',
    'authoritative_AOI': 'projects/<YOUR-EE-PROJECT>/assets/<YOUR-AEZ26-ASSET>',
    'ERA5_collection': 'ECMWF/ERA5_LAND/DAILY_AGGR',
    'CHIRPS_collection': 'UCSB-CHG/CHIRPS/DAILY',
    'role': 'extended_historical_robustness_only'
}
(OUT/'AEZ26_Robustness_Environment_and_Hashes.json').write_text(
    json.dumps(environment, indent=2)
)

# Machine-readable / human-readable method lock.
method_config = f'''AEZ-26 Phase7A Extended Historical Robustness
Status gate: reconstructed GLDAS v3 primary must equal 39 events
GLDAS pentad SHA256: {sha256_file(GLDAS_PENTAD_FILE)}
ERA5-Land period: 1981-2025
Source-only comparison period: 2000-2025
ERA5 source-only events: {len(cat_source)}
ERA5 long-record events: {len(cat_long)}
ERA5 long-record overlap events, 2000-2025: {int(cat_long["onset_year"].between(2000,2025).sum())}
No threshold tuning was performed.
A stale/superseded same-named GLDAS event CSV is ignored unless it exactly reproduces the reconstructed locked 39-event chronology.
'''
(OUT/'AEZ26_Phase7A_Method_Config.txt').write_text(
    method_config, encoding='utf-8'
)

# Independent-audit-ready summary generated from the corrected statistics.
def agreement_value(comp, tol):
    r = agreement.loc[
        (agreement['comparison'] == comp) &
        (agreement['tolerance_pentads'] == tol)
    ].iloc[0]
    return int(r['matched_primary_events']), float(r['primary_match_fraction'])

s0n, s0f = agreement_value('GLDAS_primary_vs_ERA5_source26', 0)
s1n, s1f = agreement_value('GLDAS_primary_vs_ERA5_source26', 1)
s2n, s2f = agreement_value('GLDAS_primary_vs_ERA5_source26', 2)
l0n, l0f = agreement_value('GLDAS_primary_vs_ERA5_long45_overlap', 0)
l1n, l1f = agreement_value('GLDAS_primary_vs_ERA5_long45_overlap', 1)
l2n, l2f = agreement_value('GLDAS_primary_vs_ERA5_long45_overlap', 2)

audit_notes = f'''# AEZ-26 Phase7A corrected robustness summary

## Input guard
- Locked GLDAS Phase-2C v3 chronology reconstructed from locked pentad table: **{len(g_events)} events**.
- Same-named event CSV candidates were diagnostic only; stale/superseded candidates were ignored.

## ERA5-Land sensitivity catalogues
- Same-climatology source-only ERA5 catalogue, 2000-2025: **{len(cat_source)} events**.
- Long-record ERA5 catalogue, 1981-2025: **{len(cat_long)} events**.
- Long-record ERA5 onsets during 2000-2025: **{int(cat_long["onset_year"].between(2000,2025).sum())} events**.

## One-to-one onset agreement against locked GLDAS v3 (n=39)
- ERA5 source-only: exact **{s0n}/39 ({s0f:.1%})**; ±1 pentad **{s1n}/39 ({s1f:.1%})**; ±2 pentads **{s2n}/39 ({s2f:.1%})**.
- ERA5 long-climatology overlap: exact **{l0n}/39 ({l0f:.1%})**; ±1 pentad **{l1n}/39 ({l1f:.1%})**; ±2 pentads **{l2n}/39 ({l2f:.1%})**.

## Interpretation boundary
This branch supports aggregate/seasonal robustness assessment but does not demonstrate event-by-event source invariance. The locked GLDAS 39-event chronology remains primary. ERA5-Land remains a sensitivity catalogue. No thresholds are tuned to force agreement.
'''
(OUT/'AEZ26_Phase7A_Corrected_Robustness_Summary.md').write_text(
    audit_notes, encoding='utf-8'
)

# Bundle outputs for independent audit.
zip_path = shutil.make_archive(
    str(PHASE7/'AEZ26_Phase7A_Extended_Historical_Robustness_Output_Package'),
    'zip',
    root_dir=PHASE7,
    base_dir='Phase7A_FINAL'
)
print('ROBUSTNESS PACKAGE:', zip_path)


## Interpretation rules after execution

Use this branch to answer:

- Does an independent ERA5-Land upper-1-m soil-water representation reproduce the **aggregate event-rate and seasonal context** of the locked GLDAS inventory?
- What fraction of locked GLDAS onsets agree exactly / within ±1 / within ±2 pentads under source sensitivity?
- Does the longer 1981–2025 ERA5-Land record preserve broad seasonality and event-rate context?
- What long-record hydroclimate trends are supported, and which are not?

### Required wording discipline
A similar total event count does **not** imply event-by-event invariance. If onset agreement is only low-to-moderate, report that explicitly as **source sensitivity in specific event timing**.

Do **not**:
- replace the locked 39-event GLDAS catalogue;
- use or describe a stale 40-event same-named CSV as the final Phase-2C catalogue;
- tune thresholds to improve agreement;
- call ERA5-Land and GLDAS soil-water storage numerically interchangeable;
- infer grid-cell spatial flash-drought extent from these regional means;
- claim a long-record increasing flash-drought trend unless the formal diagnostics support it;
- call ERA5-Land potential evaporation FAO-56 ET0.
